In [2]:
!pip install catboost

  Using cached catboost-1.2.10-cp313-cp313-win_amd64.whl.metadata (1.5 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/100.2 MB 2.5 MB/s eta 0:00:41
   ---------------------------------------- 1.0/100.2 MB 2.8 MB/s eta 0:00:36
    --------------------------------------- 1.8/100.2 MB 3.1 MB/s eta 0:00:33
   - -------------------------------------- 2.6/100.2 MB 3.3 MB/s eta 0:00:30
   - -------------------------------------- 3.4/100.2 MB 3.5 MB/s eta 0:00:28
   - -------------------------------------- 4.2/100.2 MB 3.6 MB/s eta 0:00:27
   - -------------------------------------- 5.0/100.2 MB 3.5 MB/s eta 0:00:27
   -- ------------------------------------- 6.0/100.2 MB 3.7 MB/s eta 0:00:26
   -- ------------------------------------- 6.3/100.2 MB 3.7 MB/s eta 0:00:26
   -- ------------

In [3]:
%run encode.ipynb

         asin                                              title  price_eur  \
0  B0G71BJS8S  Microsoft Surface Laptop, 13.8" | Snapdragon X...    1056.07   
1  B0DYDVFVTJ  Microsoft Surface Laptop, 13" | Snapdragon X P...     735.54   
2  B0F8L98RLY  HP Laptop | 15,6" FHD Display | Intel N100 | 4...     219.00   
3  B0H28V9JGR  Laptop 15,6 Zoll 8GB RAM 256GB SSD Notebook Fu...     249.19   
4  B0DJBQ8Y7K  HP Chromebook x360 Laptop (14" HD Touchscreen,...     219.00   

   rating  reviews  ram_gb  storage_gb  screen_size_inch  \
0     4.8     15.0    16.0       512.0              13.8   
1     4.3     74.0    16.0       256.0              13.0   
2     4.1    170.0     4.0       128.0              15.6   
3     4.2      8.0     8.0       256.0              15.6   
4     4.2    181.0     4.0       128.0              14.0   

                                         product_url  search_page  ...  \
0                                                NaN          1.0  ...   
1               

In [4]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# X و y
X = df_encoded.drop(
    columns=["price_eur", "asin", "title", "product_url", "search_page"],
    errors="ignore"
)

y = df_encoded["price_eur"]

# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# CatBoost
model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)

# آموزش
model.fit(X_train, y_train)

# پیش‌بینی
y_pred = model.predict(X_test)

# ارزیابی
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("R²:", r2)
print("R² (%):", r2 * 100)
print("MAE (€):", mae)
print("RMSE (€):", rmse)

R²: 0.6267854301274673
R² (%): 62.67854301274674
MAE (€): 290.4372843087446
RMSE (€): 448.35644180862096


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# X و y
x = df_encoded.drop(
    columns=["price_eur", "asin", "title", "product_url", "search_page"],
    errors="ignore"
)

y = df_encoded["price_eur"]

# تقسیم داده
X_train, X_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42
)


model_fs = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model_fs.fit(X_train, y_train)

# Feature Importance
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model_fs.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)


selected_features = importance.head(20)["feature"].tolist()

print("Selected Features:")
print(selected_features)

Selected Features:
['ram_gb', 'storage_gb', 'gpu_NVIDIA GeForce RTX 5080', 'resolution_2560x1600', 'cpu_Intel Core Ultra 9', 'resolution_1920x1080', 'screen_size_inch', 'gpu_NVIDIA GeForce RTX 5090', 'brand_MSI', 'gpu_NVIDIA GeForce RTX 5070Ti', 'reviews', 'cpu_AMD Ryzen AI 9HX', 'rating', 'gpu_NVIDIA GeForce RTX 5070', 'os_Windows 11 Home', 'gpu_NVIDIA GeForce RTX 4070', 'keyboard_AZERTY', 'gaming', 'cpu_Intel Core Ultra 9 275HX', 'os_Windows 11 Pro']


In [7]:
# X و y
X = df_encoded[selected_features]

y = df_encoded["price_eur"]

# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# CatBoost
model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)

# آموزش
model.fit(X_train, y_train)

# پیش‌بینی
y_pred = model.predict(X_test)

# ارزیابی
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("R²:", r2)
print("R² (%):", r2 * 100)
print("MAE (€):", mae)
print("RMSE (€):", rmse)

R²: 0.5579961323688114
R² (%): 55.79961323688114
MAE (€): 346.00445313321444
RMSE (€): 487.9295872283125


In [8]:
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# تعداد ویژگی‌هایی که می‌خواهیم تست کنیم
feature_counts = [20, 30, 40, 50]

results_catboost = []

for n in feature_counts:

    # انتخاب n ویژگی برتر
    selected_features = importance.head(n)["feature"].tolist()

    X = df_encoded[selected_features]
    y = df_encoded["price_eur"]

    # همان Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    # CatBoost
    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False
    )

    # آموزش
    model.fit(X_train, y_train)

    # پیش‌بینی
    y_pred = model.predict(X_test)

    # ارزیابی
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    results_catboost.append({
        "Number of Features": n,
        "R²": r2,
        "R² (%)": r2 * 100,
        "MAE (€)": mae,
        "RMSE (€)": rmse
    })

# نمایش نتایج
results_catboost = pd.DataFrame(results_catboost)

print(results_catboost)

   Number of Features        R²     R² (%)     MAE (€)    RMSE (€)
0                  20  0.557996  55.799613  346.004453  487.929587
1                  30  0.605553  60.555258  299.260391  460.933933
2                  40  0.606900  60.690037  286.586316  460.145770
3                  50  0.581035  58.103524  305.489924  475.042937
